In [3]:
import pandas as pd
from tqdm.auto import tqdm
import numpy as np 
import re
import matplotlib.pyplot as plt

from transformers import BertModel, BertTokenizer
from torch.utils.data import Dataset
import torch
import torch.nn.functional as F
from torch import nn

## textual graph attention and text feature projection

In [4]:
text_feature_dict = torch.load("text_features_3.pt")

In [6]:
df_1=pd.read_csv("twitter_data/tweets_devset.csv")
df_2=pd.read_csv("twitter_data/tweets_testset.csv")

In [66]:
text_feature_dict.keys()

dict_keys([263046056240115712, 262995061304852481, 262979898002534400, 262996108400271360, 263018881839411200, 263364439582060545, 262927032705490944, 263321078884077568, 263111677485142017, 262977091983785985, 262989009930833920, 263129115207536640, 263091320871063552, 262990978611286016, 263070862977167360, 263270378221228033, 263229641999925249, 263184498072645632, 263001267926880258, 263363592508829697, 263058558160076801, 263286294522777601, 263010650517807104, 262976546527145984, 262971979282395136, 263259176497737728, 263404091718389760, 263283812530782208, 263263438988533760, 263047501433688064, 263033265336754176, 263110586022379520, 263110236175466497, 263235193593290752, 263049636216983552, 263422787513901056, 263060279586336768, 263183410581893121, 262974263923994625, 262983606132162560, 262994195504041985, 263066368713318400, 262976505917882368, 263053746639097856, 263079860778459136, 263018516083523584, 263043396162187264, 263257996799406080, 263163316266991618, 263092194

In [7]:
text_feature_dict[df_1["tweetId"].iloc[89]]["x"].shape

torch.Size([6, 768])

In [8]:
from torch_geometric.nn import GATConv,global_mean_pool
import torch.nn.functional as F

class ResidualTextGAT(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.gat1 = GATConv(768, 768, heads=1, concat=False)
        self.gat2 = GATConv(768, 768, heads=1, concat=False)

    def forward(self, x, edge_index):
        # -------------------------
        # Layer 1 (Residual)
        # -------------------------
        h = self.gat1(x, edge_index)
        x = x + h   # 🔥 residual enrichment
        
        x = F.elu(x)

        # -------------------------
        # Layer 2 (Residual)
        # -------------------------
        h = self.gat2(x, edge_index)
        x = x + h   # 🔥 again enrich
        
        return x #[num_nodes,768]

In [9]:
from torch.utils.data import Dataset

class TextGraphDataset(Dataset):
    def __init__(self, feature_dict):
        self.data = list(feature_dict.items())

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tweet_id, item = self.data[idx]

        return {
            "tweet_id": tweet_id,
            "global": item["global"],
            "x": item["x"],
            "edge_index": item["edge_index"]
        }

In [10]:
def collate_fn(batch):
    globals_ = []
    xs = []
    edge_indices = []
    batch_index = []
    tweet_ids=[]

    node_offset = 0

    for i, item in enumerate(batch):
        x = item["x"]
        edge_index = item["edge_index"]

        num_nodes = x.size(0)

        globals_.append(item["global"])
        xs.append(x)

        if edge_index.numel() > 0:
            edge_indices.append(edge_index + node_offset)
            
        batch_index.extend([i] * num_nodes)
        tweet_ids.append(item["tweet_id"])

        node_offset += num_nodes

    x = torch.cat(xs, dim=0)

    if edge_indices:
        edge_index = torch.cat(edge_indices, dim=1)
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    batch_index = torch.tensor(batch_index)

    globals_ = torch.stack(globals_)

    return {
        "x": x,
        "edge_index": edge_index,
        "batch": batch_index,
        "global": globals_,
        "tweet_ids": tweet_ids
    }

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
from torch.utils.data import DataLoader
dataset = TextGraphDataset(text_feature_dict)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

In [18]:
class TextProjection(nn.Module):
    def __init__(self, in_dim=768, out_dim=256):
        super().__init__()
        
        self.proj = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        return self.proj(x)

In [68]:
text_proj = TextProjection().to(device)
gat_model=ResidualTextGAT().to(device)

In [69]:
text_embeddings = {}   # 🔥 final storage

for batch in loader:
    x = batch["x"].to(device)
    edge_index = batch["edge_index"].to(device)
    batch_idx = batch["batch"].to(device)
    global_feat = batch["global"].to(device)
    tweet_ids = batch["tweet_ids"]

    # GAT
    node_out = gat_model(x, edge_index)

    # Projection
    Z_text = text_proj(node_out)
    Z_text = F.normalize(Z_text, dim=1)

    # 🔥 Convert to per-tweet dict
    for i, tid in enumerate(tweet_ids):
        node_mask = (batch_idx == i)
        text_embeddings[tid] = Z_text[node_mask].detach().cpu()

In [73]:
text_embeddings[263209068909441024].shape

torch.Size([8, 256])

## image graph construction, attention and projection

In [24]:
import pickle
with open("fused_img_features.pkl", "rb") as f:
    fused_img_features = pickle.load(f)

In [ ]:
import torch.nn as nn

# ---------------------------
# Edge MLP (learns adjacency)
# ---------------------------
class EdgeMLP(nn.Module):
    def __init__(self, in_dim=512):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.mlp(x) 

In [27]:

# ---------------------------
# Graph Builder
# ---------------------------
def build_graphs(fused_img_features, edge_mlp, device, top_k=8):

    edge_mlp.eval()

    graphs = {}

    for tweet_id in fused_img_features:   # ✅ FIX: iterate over tweet_id

        patch_dict = fused_img_features[tweet_id]

        features = []
        coords = []
        patch_names = []

        # ---------------------------
        # Collect node features
        # ---------------------------
        for patch_name, data in patch_dict.items():
            features.append(data["feature"])   # (256,)
            coords.append(data["coords"])      # (x, y)
            patch_names.append(patch_name)

        # Convert to tensor
        features = torch.tensor(features, dtype=torch.float32).to(device)  # (N,256)
        coords   = torch.tensor(coords, dtype=torch.float32).to(device)    # (N,2)

        # Normalize coords (important)
        coords = coords / (coords.max() + 1e-6)

        N = features.size(0)

        # ---------------------------
        # Pairwise combinations
        # ---------------------------
        h_i = features.unsqueeze(1).repeat(1, N, 1)  # (N,N,256)
        h_j = features.unsqueeze(0).repeat(N, 1, 1)  # (N,N,256)

        pair_feat = torch.cat([h_i, h_j], dim=-1)    # (N,N,512)

        # ---------------------------
        # Learned adjacency (MLP)
        # ---------------------------
        with torch.no_grad():
            scores = edge_mlp(pair_feat).squeeze(-1)  # (N,N)

        # ---------------------------
        # Spatial bias
        # ---------------------------
        coord_i = coords.unsqueeze(1)  # (N,1,2)
        coord_j = coords.unsqueeze(0)  # (1,N,2)

        dist = torch.norm(coord_i - coord_j, dim=-1)  # (N,N)

        spatial_weight = torch.exp(-dist / 0.1)  # tune this

        scores = scores + spatial_weight

        # ---------------------------
        # Normalize (softmax over j)
        # ---------------------------
        A = torch.softmax(scores, dim=1)  # (N,N)

        # ---------------------------
        # Sparsify (top-k)
        # ---------------------------
        if top_k is not None:
            topk_vals, topk_idx = torch.topk(A, k=min(top_k, N), dim=1)

            mask = torch.zeros_like(A)
            mask.scatter_(1, topk_idx, 1)

            A = A * mask

        # ---------------------------
        # Store graph
        # ---------------------------
        graphs[tweet_id] = {
            "node_features": features.cpu(),  # (N,256)
            "adjacency": A.cpu(),             # (N,N)
            "coords": coords.cpu(),
            "patch_names": patch_names
        }

    return graphs

In [28]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model=EdgeMLP().to(device)
img_feature_graph=build_graphs(fused_img_features,model,device)

C:\Users\Anirban Banerjee\AppData\Local\Temp\ipykernel_49516\740420840.py:27: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  features = torch.tensor(features, dtype=torch.float32).to(device)  # (N,256)


In [39]:
img_feature_graph["510105654669762560"]["patch_names"]

['global_context.jpg',
 'grid_1_x0_y0.jpg',
 'grid_2_x224_y0.jpg',
 'grid_3_x224_y224.jpg',
 'obj_0_x6_y177.jpg']

In [40]:
class ImageGATLayer(nn.Module):
    def __init__(self, in_dim=256, out_dim=256):
        super().__init__()

        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, X, A):

        X_trans = self.W(X)             # (N, 256)
        agg = torch.matmul(A, X_trans) # (N, 256)

        X_out = X + agg                # residual
        X_out = self.norm(X_out)

        return X_out

In [48]:
gat_layer = ImageGATLayer().to(device)

graph = img_feature_graph["510105654669762560"]  # example tweet_id

X = graph["node_features"].to(device)   # (N,256)
A = graph["adjacency"].to(device)       # (N,N)

X_out = gat_layer(X, A)

In [45]:
def run_gat_on_graphs(img_feature_graph, gat_model, device):

    gat_model.eval()

    updated_graphs = {}

    for tweet_id, graph in img_feature_graph.items():

        X = graph["node_features"].to(device)   # (N, 256)
        A = graph["adjacency"].to(device)       # (N, N)

        with torch.no_grad():
            X_out = gat_model(X, A)             # (N, 256)

        # store updated features (keep structure same)
        updated_graphs[tweet_id] = {
            "node_features": X_out.cpu(),
            "adjacency": graph["adjacency"],   # unchanged
            "coords": graph["coords"],
            "patch_names": graph["patch_names"]
        }

    return updated_graphs

In [49]:
gat_model = ImageGATLayer().to(device)

updated_graphs = run_gat_on_graphs(
    img_feature_graph,
    gat_model,
    device
)

In [55]:
updated_graphs["510105654669762560"]["node_features"]

tensor([[-0.8320,  1.0018,  0.4000,  ..., -0.7120, -2.3644, -0.2257],
        [-1.4528,  0.6103,  0.4792,  ...,  0.1433, -2.1747, -0.4230],
        [-1.1953,  0.7619,  1.6170,  ..., -0.3668, -2.0875, -0.2839],
        [-2.7653,  0.0286,  0.8855,  ...,  0.9604, -0.6987, -0.6126],
        [-0.2543,  0.3062, -2.4179,  ..., -0.4324, -0.3295,  0.4540]])

In [51]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim=256):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )

    def forward(self, x):
        return self.proj(x)

In [ ]:
def get_image_embeddings(updated_graphs, projector, device):

    projector.eval()

    image_embeddings = {}

    for tweet_id, graph in updated_graphs.items():

        X = graph["node_features"].to(device)  # (N,256)

        # 🔥 Step 1: Pool
        img_feat = X.mean(dim=0)  # (256,)

        # 🔥 Step 2: Project
        with torch.no_grad():
            img_emb = projector(img_feat)

        image_embeddings[tweet_id] = img_emb.cpu().numpy()

    return image_embeddings

In [57]:
proj_head=ProjectionHead(in_dim=256, out_dim=256).to(device)
img_embedding = get_image_embeddings(updated_graphs, proj_head, device) 

In [60]:
img_embedding.keys()

dict_keys(['261572988766396416', '261584073011642368', '261613257507352576', '261629752052424704', '261842230443143168', '262532652647735298', '262549485908021248', '262572385356627968', '262583978568081409', '262588167918612480', '262636600092135426', '262637123272859649', '262637126234017792', '262637192063627264', '262637213853052928', '262637519319998464', '262637624597032960', '262639208601120768', '262640238919303168', '262640880282910723', '262641550209732608', '262652323778658306', '262652976613715968', '262654241871642624', '262654766415482881', '262655860831051778', '262656070340710401', '262657291298406403', '262657561000542208', '262658316264689664', '262664622493032448', '262665116036759552', '262665688383102979', '262667746385461248', '262668384087457792', '262671369337200641', '262676481971482628', '262680617865580544', '262682116721111041', '262682284644261889', '262684073997238272', '262684881597915136', '262688258335576064', '262693182737891328', '262693953969725440',